# Notebook 07 — Disruption Scenario Analysis

## Objective
Evaluate the resilience of a hypothetical hardware
supply network under operational disruptions.

## Scenarios
1. Primary supplier order interruption
2. Regional demand surge
3. Manufacturing capacity reduction
4. Standard freight cost inflation

## Methodology
Rebuild and solve the original 12-week Pyomo
optimization model under modified assumptions.

For every scenario:
- Preserve the original network structure.
- Apply only the specified disruption.
- Reoptimize operational decisions.
- Maintain full demand fulfillment where feasible.
- Preserve original terminal inventory targets.
- Independently validate the resulting supply plan.

## Key Outputs
- Scenario feasibility and optimal cost
- Incremental cost versus reference optimization
- Changes in sourcing and production
- Transportation mode decisions
- Inventory balances and capacity utilization
- Executive scenario comparison

## Limitations
All inputs are hypothetical and deterministic.
Results represent modeled scenarios rather than
real-world performance forecasts.


In [1]:
# STEP 2: Initialize disruption scenario notebook

import pandas as pd
import numpy as np
import pyomo.environ as pyo
from pathlib import Path
from pyomo.opt import SolverStatus, TerminationCondition
from IPython.display import display, Markdown

# Locate the repository
current_path = Path.cwd().resolve()

repo_root = next(
    (
        p for p in [current_path, *current_path.parents]
        if (p / "data/assumptions/bom.csv").is_file()
    ),
    None
)

assert repo_root is not None, "Project 2 repository not found"

# Load network assumptions, demand and baseline targets
files = {
    "bom": "data/assumptions/bom.csv",
    "suppliers": "data/assumptions/suppliers.csv",
    "plants": "data/assumptions/plants.csv",
    "dcs": "data/assumptions/distribution_centers.csv",
    "lanes": "data/assumptions/transport_lanes.csv",
    "inventory": "data/assumptions/initial_inventory.csv",
    "demand": "data/processed/regional_demand_12_weeks.csv",

    "baseline_components":
        "outputs/tables/baseline_component_inventory.csv",

    "baseline_plant":
        "outputs/tables/baseline_plant_inventory.csv",

    "baseline_dcs":
        "outputs/tables/baseline_dc_inventory.csv",

    "optimized_cost":
        "outputs/tables/optimized_cost_breakdown.csv"
}

data = {}

for name, relative_path in files.items():
    path = repo_root / relative_path
    assert path.is_file(), f"Missing file: {path}"
    data[name] = pd.read_csv(path)

assert len(data) == 11

reference_cost = float(
    data["optimized_cost"]["Cost_USD"].sum()
)

assert reference_cost > 0
assert data["demand"]["Demand_Units"].sum() == 6180

print("NOTEBOOK 07 INITIALIZED SUCCESSFULLY")
print("=" * 45)
print("Datasets loaded:", len(data))
print("Planning weeks:", data["demand"]["Week"].nunique())
print("Reference optimized cost: $",
      round(reference_cost, 2))
print("Customer demand:",
      data["demand"]["Demand_Units"].sum())

display(pd.DataFrame([
    {"Dataset": name, "Records": len(df)}
    for name, df in data.items()
]))


NOTEBOOK 07 INITIALIZED SUCCESSFULLY
Datasets loaded: 11
Planning weeks: 12
Reference optimized cost: $ 404131.2
Customer demand: 6180


,Dataset,Records
0,bom,3
1,suppliers,6
2,plants,1
3,dcs,2
4,lanes,4
5,inventory,6
6,demand,24
7,baseline_components,36
8,baseline_plant,12
9,baseline_dcs,24


In [2]:
# STEP 3: Define hypothetical disruption scenarios

scenarios = [
    {
        "Scenario": "Reference",
        "Type": "reference",
        "Description": "Original optimized network"
    },
    {
        "Scenario": "Supplier_Disruption",
        "Type": "supplier_outage",
        "Supplier": "SUP_A",
        "Affected_Weeks": [4, 5],
        "Description": "Supplier A accepts no new orders in Weeks 4-5"
    },
    {
        "Scenario": "Demand_Surge",
        "Type": "demand_surge",
        "Affected_Weeks": [6, 7, 8],
        "Demand_Increase_Pct": 10,
        "Description": "Regional demand increases 10% in Weeks 6-8"
    },
    {
        "Scenario": "Manufacturing_Disruption",
        "Type": "plant_reduction",
        "Affected_Weeks": [4, 5, 6, 7],
        "Capacity_Reduction_Pct": 20,
        "Description": "Plant capacity decreases 20% in Weeks 4-7"
    },
    {
        "Scenario": "Freight_Inflation",
        "Type": "freight_increase",
        "Transport_Mode": "Standard",
        "Cost_Increase_Pct": 50,
        "Description": "Standard freight cost increases 50%"
    }
]

scenario_summary = pd.DataFrame([
    {
        "Scenario": s["Scenario"],
        "Scenario_Type": s["Type"],
        "Description": s["Description"]
    }
    for s in scenarios
])

assert len(scenarios) == 5
assert scenario_summary["Scenario"].is_unique

print("DISRUPTION SCENARIOS DEFINED")
display(scenario_summary)

print("\nFour disruption scenarios + reference")
print("STEP 3: PASSED")


DISRUPTION SCENARIOS DEFINED


,Scenario,Scenario_Type,Description
0,Reference,reference,Original optimized network
1,Supplier_Disruption,supplier_outage,Supplier A accepts no new orders in Weeks 4-5
2,Demand_Surge,demand_surge,Regional demand increases 10% in Weeks 6-8
3,Manufacturing_Disruption,plant_reduction,Plant capacity decreases 20% in Weeks 4-7
4,Freight_Inflation,freight_increase,Standard freight cost increases 50%



Four disruption scenarios + reference
STEP 3: PASSED


In [3]:
# STEP 4A: Prepare reusable optimization parameters

bom_df = data["bom"]
supplier_df = data["suppliers"]
plant_df = data["plants"]
dc_df = data["dcs"]
lane_df = data["lanes"]
inventory_df = data["inventory"]

weeks = sorted(data["demand"]["Week"].astype(int).unique())
components = bom_df["Component_ID"].tolist()
dcs = dc_df["DC_ID"].tolist()
lanes = lane_df["Lane_ID"].tolist()

eligible = supplier_df.loc[
    supplier_df["Eligible"].astype(str).str.lower() == "true"
].copy()

pairs = list(
    eligible[["Supplier_ID", "Component_ID"]]
    .itertuples(index=False, name=None)
)

bom_qty = dict(zip(
    bom_df["Component_ID"],
    bom_df["Units_Per_Finished_Unit"]
))

supplier_cost = {
    (r.Supplier_ID, r.Component_ID): float(r.Unit_Cost_USD)
    for r in eligible.itertuples(index=False)
}

supplier_capacity = {
    (r.Supplier_ID, r.Component_ID): int(r.Weekly_Capacity_Units)
    for r in eligible.itertuples(index=False)
}

supplier_lead = {
    (r.Supplier_ID, r.Component_ID): int(r.Lead_Time_Weeks)
    for r in eligible.itertuples(index=False)
}

component_suppliers = {
    c: [s for s, component in pairs if component == c]
    for c in components
}

lane_cost = dict(zip(
    lane_df["Lane_ID"],
    lane_df["Cost_Per_Unit_USD"]
))

lane_capacity = dict(zip(
    lane_df["Lane_ID"],
    lane_df["Weekly_Lane_Capacity_Units"]
))

lane_lead = dict(zip(
    lane_df["Lane_ID"],
    lane_df["Transit_Time_Weeks"]
))

lane_dest = dict(zip(
    lane_df["Lane_ID"],
    lane_df["Destination_DC_ID"]
))

lanes_by_dc = {
    d: [l for l in lanes if lane_dest[l] == d]
    for d in dcs
}

dc_capacity = dict(zip(
    dc_df["DC_ID"],
    dc_df["Storage_Capacity_Units"]
))

dc_holding = dict(zip(
    dc_df["DC_ID"],
    dc_df["Holding_Cost_Per_Unit_Per_Week_USD"]
))

base_demand = {
    (r.DC_ID, int(r.Week)): int(r.Demand_Units)
    for r in data["demand"].itertuples(index=False)
}

plant_capacity = int(plant_df["Weekly_Capacity_Units"].iloc[0])
assembly_cost = float(
    plant_df["Assembly_Cost_Per_Unit_USD"].iloc[0]
)

initial_components = (
    inventory_df.loc[
        inventory_df["Item_Type"] == "Component"
    ].set_index("Item_ID")["Beginning_Inventory_Units"]
    .to_dict()
)

initial_finished = (
    inventory_df.loc[
        inventory_df["Item_Type"] == "Finished_Good"
    ].set_index("Location_ID")["Beginning_Inventory_Units"]
    .to_dict()
)

plant_id = plant_df["Plant_ID"].iloc[0]
last_week = max(weeks)

# Match the original baseline terminal inventory
terminal_components = (
    data["baseline_components"].loc[
        data["baseline_components"]["Week"] == last_week
    ].set_index("Component_ID")["Ending_Inventory"]
    .to_dict()
)

terminal_plant = float(
    data["baseline_plant"].loc[
        data["baseline_plant"]["Week"] == last_week,
        "Ending_Inventory"
    ].iloc[0]
)

terminal_dc = (
    data["baseline_dcs"].loc[
        data["baseline_dcs"]["Week"] == last_week
    ].set_index("DC_ID")["Ending_Inventory"]
    .to_dict()
)

assert weeks == list(range(1, 13))
assert len(pairs) == 6
assert len(lanes) == 4
assert len(dcs) == 2
assert set(initial_components) == set(components)
assert set(terminal_dc) == set(dcs)

print("REUSABLE OPTIMIZATION PARAMETERS READY")
print("Weeks:", len(weeks))
print("Supplier-component combinations:", len(pairs))
print("Original demand:", sum(base_demand.values()))
print("STEP 4A: PASSED")


REUSABLE OPTIMIZATION PARAMETERS READY
Weeks: 12
Supplier-component combinations: 6
Original demand: 6180
STEP 4A: PASSED


In [4]:
# STEP 4B: Build reusable Pyomo scenario model

def build_scenario_model(spec):

    scenario_type = spec["Type"]

    # 1. Copy original assumptions
    supply_cap = {
        (s, c, t): supplier_capacity[s, c]
        for s, c in pairs
        for t in weeks
    }

    production_cap = {
        t: plant_capacity for t in weeks
    }

    demand = base_demand.copy()
    freight_cost = lane_cost.copy()

    # 2. Apply scenario-specific changes
    if scenario_type == "supplier_outage":
        for s, c in pairs:
            if s == spec["Supplier"]:
                for t in spec["Affected_Weeks"]:
                    supply_cap[s, c, t] = 0

    elif scenario_type == "plant_reduction":
        reduction = spec["Capacity_Reduction_Pct"] / 100
        for t in spec["Affected_Weeks"]:
            production_cap[t] = int(
                np.floor(plant_capacity * (1 - reduction))
            )

    elif scenario_type == "demand_surge":
        increase = spec["Demand_Increase_Pct"] / 100
        for d in dcs:
            for t in spec["Affected_Weeks"]:
                demand[d, t] = int(round(
                    base_demand[d, t] * (1 + increase)
                ))

    elif scenario_type == "freight_increase":
        increase = spec["Cost_Increase_Pct"] / 100
        for row in lane_df.itertuples(index=False):
            if row.Transport_Mode == spec["Transport_Mode"]:
                freight_cost[row.Lane_ID] *= (1 + increase)

    elif scenario_type != "reference":
        raise ValueError(f"Unknown scenario: {scenario_type}")

    # 3. Initialize model
    m = pyo.ConcreteModel(name=spec["Scenario"])

    m.T = pyo.Set(initialize=weeks, ordered=True)
    m.C = pyo.Set(initialize=components)
    m.D = pyo.Set(initialize=dcs)
    m.L = pyo.Set(initialize=lanes)
    m.SC = pyo.Set(dimen=2, initialize=pairs)

    # 4. Decision variables
    m.purchase = pyo.Var(
        m.SC, m.T, domain=pyo.NonNegativeIntegers
    )

    m.production = pyo.Var(
        m.T, domain=pyo.NonNegativeIntegers
    )

    m.ship = pyo.Var(
        m.L, m.T, domain=pyo.NonNegativeIntegers
    )

    m.component_inv = pyo.Var(
        m.C, m.T, domain=pyo.NonNegativeIntegers
    )

    m.plant_inv = pyo.Var(
        m.T, domain=pyo.NonNegativeIntegers
    )

    m.dc_inv = pyo.Var(
        m.D, m.T, domain=pyo.NonNegativeIntegers
    )

    # 5. Objective cost expressions
    m.procurement_cost = pyo.Expression(
        expr=sum(
            supplier_cost[s, c] * m.purchase[s, c, t]
            for s, c in pairs for t in weeks
        )
    )

    m.manufacturing_cost = pyo.Expression(
        expr=sum(
            assembly_cost * m.production[t]
            for t in weeks
        )
    )

    m.freight_cost = pyo.Expression(
        expr=sum(
            freight_cost[l] * m.ship[l, t]
            for l in lanes for t in weeks
        )
    )

    m.holding_cost = pyo.Expression(
        expr=sum(
            dc_holding[d] * m.dc_inv[d, t]
            for d in dcs for t in weeks
        )
    )

    m.total_cost = pyo.Objective(
        expr=(
            m.procurement_cost
            + m.manufacturing_cost
            + m.freight_cost
            + m.holding_cost
        ),
        sense=pyo.minimize
    )

    # 6. Supplier weekly order capacities
    m.supplier_limit = pyo.Constraint(
        m.SC, m.T,
        rule=lambda m, s, c, t:
            m.purchase[s, c, t] <= supply_cap[s, c, t]
    )

    # 7. Component inventory balances
    def component_balance(m, c, t):

        beginning = (
            initial_components[c]
            if t == weeks[0]
            else m.component_inv[c, t - 1]
        )

        receipts = sum(
            m.purchase[s, c, t - supplier_lead[s, c]]
            for s in component_suppliers[c]
            if (t - supplier_lead[s, c]) in m.T
        )

        consumption = bom_qty[c] * m.production[t]

        return (
            beginning + receipts - consumption
            == m.component_inv[c, t]
        )

    m.component_balance = pyo.Constraint(
        m.C, m.T, rule=component_balance
    )

    # 8. No purchases arriving beyond Week 12
    def purchase_horizon(m, s, c, t):
        if t + supplier_lead[s, c] > last_week:
            return m.purchase[s, c, t] == 0
        return pyo.Constraint.Skip

    m.purchase_horizon = pyo.Constraint(
        m.SC, m.T, rule=purchase_horizon
    )

    # 9. Manufacturing capacity
    m.production_limit = pyo.Constraint(
        m.T,
        rule=lambda m, t:
            m.production[t] <= production_cap[t]
    )

    # 10. Plant finished goods inventory
    def plant_balance(m, t):

        beginning = (
            initial_finished[plant_id]
            if t == weeks[0]
            else m.plant_inv[t - 1]
        )

        return (
            beginning
            + m.production[t]
            - sum(m.ship[l, t] for l in m.L)
            == m.plant_inv[t]
        )

    m.plant_balance = pyo.Constraint(
        m.T, rule=plant_balance
    )

    # 11. Transportation capacity
    m.transport_limit = pyo.Constraint(
        m.L, m.T,
        rule=lambda m, l, t:
            m.ship[l, t] <= lane_capacity[l]
    )

    # 12. No shipments arriving beyond Week 12
    def transport_horizon(m, l, t):
        if t + lane_lead[l] > last_week:
            return m.ship[l, t] == 0
        return pyo.Constraint.Skip

    m.transport_horizon = pyo.Constraint(
        m.L, m.T, rule=transport_horizon
    )

    # Calculate transportation arrivals
    def arrivals(m, d, t):
        return sum(
            m.ship[l, t - lane_lead[l]]
            for l in lanes_by_dc[d]
            if (t - lane_lead[l]) in m.T
        )

    # 13. Distribution inventory balance
    def dc_balance(m, d, t):

        beginning = (
            initial_finished[d]
            if t == weeks[0]
            else m.dc_inv[d, t - 1]
        )

        return (
            beginning + arrivals(m, d, t)
            - demand[d, t]
            == m.dc_inv[d, t]
        )

    m.dc_balance = pyo.Constraint(
        m.D, m.T, rule=dc_balance
    )

    # 14. Storage capacity before demand
    def dc_storage(m, d, t):

        beginning = (
            initial_finished[d]
            if t == weeks[0]
            else m.dc_inv[d, t - 1]
        )

        expr = (
            beginning + arrivals(m, d, t)
            <= dc_capacity[d]
        )

        # Handle constant Week 1 constraints
        if isinstance(expr, (bool, np.bool_)):
            if not expr:
                raise ValueError(
                    f"Initial storage exceeds capacity: {d}"
                )
            return pyo.Constraint.Feasible

        return expr

    m.dc_storage = pyo.Constraint(
        m.D, m.T, rule=dc_storage
    )

    # 15. Same terminal inventory as Notebook 04
    m.terminal_components = pyo.Constraint(
        m.C,
        rule=lambda m, c:
            m.component_inv[c, last_week]
            == terminal_components[c]
    )

    m.terminal_plant = pyo.Constraint(
        expr=m.plant_inv[last_week] == terminal_plant
    )

    m.terminal_dc = pyo.Constraint(
        m.D,
        rule=lambda m, d:
            m.dc_inv[d, last_week] == terminal_dc[d]
    )

    # Scenario-specific data needed for auditing
    parameters = {
        "Supply_Capacity": supply_cap,
        "Plant_Capacity": production_cap,
        "Demand": demand,
        "Freight_Cost": freight_cost
    }

    return m, parameters


# Verify the scenario model constructs correctly
reference_model, reference_params = (
    build_scenario_model(scenarios[0])
)

variable_count = sum(
    len(v) for v in reference_model.component_objects(pyo.Var)
)

constraint_count = sum(
    1 for _ in reference_model.component_data_objects(
        pyo.Constraint, active=True
    )
)

assert variable_count == 204
assert constraint_count == 247

print("REUSABLE OPTIMIZATION MODEL CREATED")
print("Decision variables:", variable_count)
print("Active constraints:", constraint_count)
print("\nSTEP 4B: PASSED")


REUSABLE OPTIMIZATION MODEL CREATED
Decision variables: 204
Active constraints: 247

STEP 4B: PASSED


In [5]:
# STEP 5A: Solve scenarios and extract results

solver = pyo.SolverFactory("highs")
assert solver.available(exception_flag=False)

def integer_result(variable):
    value = float(pyo.value(variable))
    assert np.isfinite(value)
    assert abs(value - round(value)) < 1e-5
    return int(round(value))


def solve_scenario(spec):

    m, params = build_scenario_model(spec)
    name = spec["Scenario"]

    try:
        results = solver.solve(
            m,
            tee=False,
            load_solutions=False
        )
    except Exception as error:
        return {
            "Scenario": name,
            "Status": "Solver_Error",
            "Message": str(error),
            "Parameters": params,
            "Details": None
        }

    termination = results.solver.termination_condition

    if termination != TerminationCondition.optimal:
        return {
            "Scenario": name,
            "Status": str(termination),
            "Message": "No verified optimal solution",
            "Parameters": params,
            "Details": None
        }

    m.solutions.load_from(results)

    # Audit every active constraint
    for constraint in m.component_data_objects(
        pyo.Constraint, active=True
    ):
        body = pyo.value(constraint.body)

        if constraint.lower is not None:
            assert body >= (
                pyo.value(constraint.lower) - 1e-5
            ), constraint.name

        if constraint.upper is not None:
            assert body <= (
                pyo.value(constraint.upper) + 1e-5
            ), constraint.name

    # Extract purchasing decisions
    purchases = pd.DataFrame([
        {
            "Scenario": name,
            "Supplier_ID": s,
            "Component_ID": c,
            "Order_Week": t,
            "Arrival_Week": t + supplier_lead[s, c],
            "Order_Units": integer_result(
                m.purchase[s, c, t]
            ),
            "Unit_Cost_USD": supplier_cost[s, c]
        }
        for s, c in pairs
        for t in weeks
    ])

    purchases["Purchase_Cost_USD"] = (
        purchases["Order_Units"]
        * purchases["Unit_Cost_USD"]
    )

    # Extract production decisions
    production = pd.DataFrame([
        {
            "Scenario": name,
            "Week": t,
            "Production_Units": integer_result(
                m.production[t]
            ),
            "Available_Capacity": params[
                "Plant_Capacity"
            ][t]
        }
        for t in weeks
    ])

    # Extract shipment decisions
    shipments = pd.DataFrame([
        {
            "Scenario": name,
            "Lane_ID": l,
            "DC_ID": lane_dest[l],
            "Ship_Week": t,
            "Arrival_Week": t + lane_lead[l],
            "Ship_Units": integer_result(m.ship[l, t]),
            "Unit_Freight_Cost_USD":
                params["Freight_Cost"][l]
        }
        for l in lanes
        for t in weeks
    ])

    shipments["Freight_Cost_USD"] = (
        shipments["Ship_Units"]
        * shipments["Unit_Freight_Cost_USD"]
    )

    # Extract inventory balances
    component_inv = pd.DataFrame([
        {
            "Scenario": name,
            "Week": t,
            "Component_ID": c,
            "Ending_Inventory": integer_result(
                m.component_inv[c, t]
            )
        }
        for c in components
        for t in weeks
    ])

    plant_inv = pd.DataFrame([
        {
            "Scenario": name,
            "Week": t,
            "Ending_Inventory": integer_result(
                m.plant_inv[t]
            )
        }
        for t in weeks
    ])

    dc_inv = pd.DataFrame([
        {
            "Scenario": name,
            "Week": t,
            "DC_ID": d,
            "Customer_Demand": params["Demand"][d, t],
            "Ending_Inventory": integer_result(
                m.dc_inv[d, t]
            )
        }
        for d in dcs
        for t in weeks
    ])

    # Independently recalculate modeled costs
    procurement_cost = purchases[
        "Purchase_Cost_USD"
    ].sum()

    manufacturing_cost = (
        production["Production_Units"].sum()
        * assembly_cost
    )

    freight_cost = shipments[
        "Freight_Cost_USD"
    ].sum()

    holding_cost = sum(
        row.Ending_Inventory * dc_holding[row.DC_ID]
        for row in dc_inv.itertuples(index=False)
    )

    total_cost = (
        procurement_cost + manufacturing_cost
        + freight_cost + holding_cost
    )

    assert np.isclose(
        total_cost, pyo.value(m.total_cost), atol=0.01
    )

    return {
        "Scenario": name,
        "Status": "Optimal",
        "Message": "All model constraints satisfied",
        "Parameters": params,
        "Total_Cost_USD": float(total_cost),
        "Customer_Demand_Units": int(
            sum(params["Demand"].values())
        ),
        "Procurement_Cost_USD": float(procurement_cost),
        "Manufacturing_Cost_USD": float(manufacturing_cost),
        "Freight_Cost_USD": float(freight_cost),
        "Holding_Cost_USD": float(holding_cost),
        "Details": {
            "Purchases": purchases,
            "Production": production,
            "Shipments": shipments,
            "Component_Inventory": component_inv,
            "Plant_Inventory": plant_inv,
            "DC_Inventory": dc_inv
        }
    }


print("SCENARIO SOLVER CREATED SUCCESSFULLY")
print("Solver: HiGHS")
print("STEP 5A: PASSED")


SCENARIO SOLVER CREATED SUCCESSFULLY
Solver: HiGHS
STEP 5A: PASSED


In [6]:
# STEP 5B: Verify reference optimization reproduction

reference_result = solve_scenario(scenarios[0])

assert reference_result["Status"] == "Optimal", (
    reference_result["Message"]
)

assert np.isclose(
    reference_result["Total_Cost_USD"],
    reference_cost,
    atol=0.01
), (
    "Reference reproduction failed. Review model "
    "formulation before running disruption scenarios."
)

print("REFERENCE MODEL REPRODUCTION")
print("=" * 45)
print("Original Notebook 04 cost: $",
      round(reference_cost, 2))
print("Rebuilt model cost: $",
      round(reference_result["Total_Cost_USD"], 2))
print("Difference: $",
      round(
          reference_result["Total_Cost_USD"]
          - reference_cost, 2
      ))

print("\nREFERENCE REPRODUCTION: PASSED")
print("STEP 5B: PASSED")


REFERENCE MODEL REPRODUCTION
Original Notebook 04 cost: $ 404131.2
Rebuilt model cost: $ 404131.2
Difference: $ 0.0

REFERENCE REPRODUCTION: PASSED
STEP 5B: PASSED


In [7]:
# STEP 5C: Solve all disruption scenarios

scenario_results = [reference_result]

for spec in scenarios[1:]:

    print("\nSolving:", spec["Scenario"])

    result = solve_scenario(spec)
    scenario_results.append(result)

    print("Status:", result["Status"])

    if result["Status"] == "Optimal":
        print(
            "Optimized cost: $",
            round(result["Total_Cost_USD"], 2)
        )
        print(
            "Customer demand:",
            result["Customer_Demand_Units"]
        )
    else:
        print("Reason:", result["Message"])

print("\nSCENARIO SOLVER RUNS COMPLETE")
print("Scenarios attempted:", len(scenario_results))
print("STEP 5C: PASSED")



Solving: Supplier_Disruption
Status: Optimal
Optimized cost: $ 417631.2
Customer demand: 6180

Solving: Demand_Surge
Status: Optimal
Optimized cost: $ 418967.7
Customer demand: 6336

Solving: Manufacturing_Disruption
Status: Optimal
Optimized cost: $ 405741.2
Customer demand: 6180

Solving: Freight_Inflation
Status: Optimal
Optimized cost: $ 413920.45
Customer demand: 6180

SCENARIO SOLVER RUNS COMPLETE
Scenarios attempted: 5
STEP 5C: PASSED


In [8]:
# STEP 6A: Create scenario comparison table

scenario_rows = []

for result in scenario_results:

    optimal = result["Status"] == "Optimal"

    cost = (
        result["Total_Cost_USD"]
        if optimal else np.nan
    )

    demand = (
        result["Customer_Demand_Units"]
        if optimal else np.nan
    )

    incremental = (
        cost - reference_cost
        if optimal else np.nan
    )

    scenario_rows.append({
        "Scenario": result["Scenario"],
        "Status": result["Status"],
        "Total_Demand_Units": demand,
        "Optimized_Cost_USD": cost,
        "Incremental_Cost_USD": incremental,
        "Cost_Change_Pct": (
            incremental / reference_cost * 100
            if optimal else np.nan
        )
    })

scenario_comparison = pd.DataFrame(scenario_rows)

assert len(scenario_comparison) == 5

reference_row = scenario_comparison.loc[
    scenario_comparison["Scenario"] == "Reference"
].iloc[0]

assert np.isclose(
    reference_row["Optimized_Cost_USD"],
    reference_cost
)

print("DISRUPTION SCENARIO COMPARISON")
print("=" * 45)

display(scenario_comparison.round(2))

print("\nSTEP 6A: PASSED")


DISRUPTION SCENARIO COMPARISON


,Scenario,Status,Total_Demand_Units,Optimized_Cost_USD,Incremental_Cost_USD,Cost_Change_Pct
0,Reference,Optimal,6180,404131.20,0.00,0.00
1,Supplier_Disruption,Optimal,6180,417631.20,13500.00,3.34
2,Demand_Surge,Optimal,6336,418967.70,14836.50,3.67
3,Manufacturing_Disruption,Optimal,6180,405741.20,1610.00,0.40
4,Freight_Inflation,Optimal,6180,413920.45,9789.25,2.42



STEP 6A: PASSED


In [9]:
# STEP 6B: Independently validate scenario inventory flows

def audit_scenario(result):

    if result["Status"] != "Optimal":
        return "NOT_APPLICABLE"

    details = result["Details"]
    params = result["Parameters"]

    orders = details["Purchases"]
    production = details["Production"]
    shipments = details["Shipments"]
    component_ledger = details["Component_Inventory"]
    plant_ledger = details["Plant_Inventory"]
    dc_ledger = details["DC_Inventory"]

    component_stock = {
        c: int(initial_components[c]) for c in components
    }

    plant_stock = int(initial_finished[plant_id])
    dc_stock = {
        d: int(initial_finished[d]) for d in dcs
    }

    # Supplier and transportation capacity compliance
    for row in orders.itertuples(index=False):
        assert row.Order_Units <= params[
            "Supply_Capacity"
        ][row.Supplier_ID, row.Component_ID, row.Order_Week]

    for row in shipments.itertuples(index=False):
        assert row.Ship_Units <= lane_capacity[row.Lane_ID]

        if row.Ship_Units > 0:
            assert row.Arrival_Week <= last_week

    for week in weeks:

        produced = int(
            production.loc[
                production["Week"] == week,
                "Production_Units"
            ].iloc[0]
        )

        assert produced <= params["Plant_Capacity"][week]

        # Reconstruct component balances
        for c in components:

            receipts = int(orders.loc[
                (orders["Component_ID"] == c)
                & (orders["Arrival_Week"] == week),
                "Order_Units"
            ].sum())

            component_stock[c] += (
                receipts - produced * bom_qty[c]
            )

            assert component_stock[c] >= 0

            recorded = component_ledger.loc[
                (component_ledger["Week"] == week)
                & (component_ledger["Component_ID"] == c),
                "Ending_Inventory"
            ].iloc[0]

            assert np.isclose(component_stock[c], recorded)

        # Reconstruct plant finished goods
        outbound = int(shipments.loc[
            shipments["Ship_Week"] == week,
            "Ship_Units"
        ].sum())

        plant_stock += produced - outbound
        assert plant_stock >= 0

        recorded_plant = plant_ledger.loc[
            plant_ledger["Week"] == week,
            "Ending_Inventory"
        ].iloc[0]

        assert np.isclose(plant_stock, recorded_plant)

        # Reconstruct distribution center inventories
        for d in dcs:

            arrivals = int(shipments.loc[
                (shipments["DC_ID"] == d)
                & (shipments["Arrival_Week"] == week),
                "Ship_Units"
            ].sum())

            available = dc_stock[d] + arrivals

            assert available <= dc_capacity[d]

            customer_demand = params["Demand"][d, week]

            # Full customer service required
            assert available >= customer_demand

            dc_stock[d] = available - customer_demand

            recorded_dc = dc_ledger.loc[
                (dc_ledger["Week"] == week)
                & (dc_ledger["DC_ID"] == d),
                "Ending_Inventory"
            ].iloc[0]

            assert np.isclose(dc_stock[d], recorded_dc)

    # Validate terminal inventory
    for c in components:
        assert np.isclose(
            component_stock[c], terminal_components[c]
        )

    assert np.isclose(plant_stock, terminal_plant)

    for d in dcs:
        assert np.isclose(dc_stock[d], terminal_dc[d])

    # Total finished goods conservation
    starting_fg = (
        initial_finished[plant_id]
        + sum(initial_finished[d] for d in dcs)
    )

    total_produced = production["Production_Units"].sum()
    total_demand = sum(params["Demand"].values())

    ending_fg = plant_stock + sum(dc_stock.values())

    assert np.isclose(
        starting_fg + total_produced,
        total_demand + ending_fg
    )

    return "PASSED"


audit_records = []

for result in scenario_results:

    status = audit_scenario(result)

    audit_records.append({
        "Scenario": result["Scenario"],
        "Solver_Status": result["Status"],
        "Independent_Audit": status
    })

scenario_audit = pd.DataFrame(audit_records)

print("SCENARIO INVENTORY FLOW AUDIT")
display(scenario_audit)

assert (
    scenario_audit.loc[
        scenario_audit["Solver_Status"] == "Optimal",
        "Independent_Audit"
    ] == "PASSED"
).all()

print("\nSTEP 6B: PASSED")


SCENARIO INVENTORY FLOW AUDIT


,Scenario,Solver_Status,Independent_Audit
0,Reference,Optimal,PASSED
1,Supplier_Disruption,Optimal,PASSED
2,Demand_Surge,Optimal,PASSED
3,Manufacturing_Disruption,Optimal,PASSED
4,Freight_Inflation,Optimal,PASSED



STEP 6B: PASSED


In [10]:
# STEP 6C: Analyze operational responses to disruptions

operational_rows = []

lane_modes = dict(zip(
    lane_df["Lane_ID"],
    lane_df["Transport_Mode"]
))

for result in scenario_results:

    if result["Status"] != "Optimal":
        continue

    details = result["Details"]
    orders = details["Purchases"]
    production = details["Production"]
    shipments = details["Shipments"]
    comp = details["Component_Inventory"]
    params = result["Parameters"]

    expedited_units = sum(
        row.Ship_Units
        for row in shipments.itertuples(index=False)
        if lane_modes[row.Lane_ID] == "Expedited"
    )

    total_shipped = shipments["Ship_Units"].sum()

    weeks_at_plant_capacity = sum(
        int(row.Production_Units ==
            params["Plant_Capacity"][row.Week])
        for row in production.itertuples(index=False)
    )

    operational_rows.append({
        "Scenario": result["Scenario"],
        "Supplier_A_Orders": orders.loc[
            orders["Supplier_ID"] == "SUP_A",
            "Order_Units"
        ].sum(),
        "Supplier_B_Orders": orders.loc[
            orders["Supplier_ID"] == "SUP_B",
            "Order_Units"
        ].sum(),
        "Production_Units":
            production["Production_Units"].sum(),
        "Peak_Weekly_Production":
            production["Production_Units"].max(),
        "Weeks_At_Plant_Capacity":
            weeks_at_plant_capacity,
        "Expedited_Shipment_Units":
            expedited_units,
        "Expedited_Share_Pct":
            expedited_units / total_shipped * 100,
        "Zero_Component_Buffer_Weeks":
            int(comp["Ending_Inventory"].eq(0).sum())
    })

scenario_operations = pd.DataFrame(operational_rows)

print("OPERATIONAL DISRUPTION RESPONSE")
display(scenario_operations.round(2))

print("\nSTEP 6C: PASSED")

OPERATIONAL DISRUPTION RESPONSE


,Scenario,Supplier_A_Orders,Supplier_B_Orders,Production_Units,Peak_Weekly_Production,Weeks_At_Plant_Capacity,Expedited_Shipment_Units,Expedited_Share_Pct,Zero_Component_Buffer_Weeks
0,Reference,12252,1443,5665,580,3,562,9.92,26
1,Supplier_Disruption,9552,4143,5665,580,6,562,9.92,18
2,Demand_Surge,12252,1911,5821,580,5,562,9.65,20
3,Manufacturing_Disruption,12567,1128,5665,580,8,1356,23.94,22
4,Freight_Inflation,12837,858,5665,580,3,1537,27.13,32



STEP 6C: PASSED


In [11]:
# STEP 7A: Export disruption analysis results

output_dir = repo_root / "outputs" / "tables"
output_dir.mkdir(parents=True, exist_ok=True)

# Collect detailed results from optimal runs only
solved = [
    r for r in scenario_results
    if r["Status"] == "Optimal"
]

def combine_details(key):
    return pd.concat(
        [r["Details"][key] for r in solved],
        ignore_index=True
    )

# Cost-category reporting
scenario_cost_rows = []

cost_fields = {
    "Procurement": "Procurement_Cost_USD",
    "Manufacturing": "Manufacturing_Cost_USD",
    "Transportation": "Freight_Cost_USD",
    "DC Inventory Holding": "Holding_Cost_USD"
}

for result in solved:
    for category, field in cost_fields.items():
        scenario_cost_rows.append({
            "Scenario": result["Scenario"],
            "Cost_Category": category,
            "Cost_USD": result[field]
        })

scenario_cost_breakdown = pd.DataFrame(
    scenario_cost_rows
)

exports = {
    "scenario_comparison.csv": scenario_comparison,
    "scenario_audit.csv": scenario_audit,
    "scenario_operational_response.csv":
        scenario_operations,
    "scenario_cost_breakdown.csv":
        scenario_cost_breakdown,
    "scenario_purchase_orders.csv":
        combine_details("Purchases"),
    "scenario_production.csv":
        combine_details("Production"),
    "scenario_shipments.csv":
        combine_details("Shipments"),
    "scenario_dc_inventory.csv":
        combine_details("DC_Inventory"),
    "scenario_component_inventory.csv":
        combine_details("Component_Inventory")
}

for filename, dataframe in exports.items():
    path = output_dir / filename
    assert len(dataframe) > 0
    dataframe.to_csv(path, index=False)
    assert path.is_file()
    print("Exported:", filename)

print("\nSCENARIO ANALYSIS EXPORT COMPLETE")
print("Files created:", len(exports))
print("STEP 7A: PASSED")


Exported: scenario_comparison.csv
Exported: scenario_audit.csv
Exported: scenario_operational_response.csv
Exported: scenario_cost_breakdown.csv
Exported: scenario_purchase_orders.csv
Exported: scenario_production.csv
Exported: scenario_shipments.csv
Exported: scenario_dc_inventory.csv
Exported: scenario_component_inventory.csv

SCENARIO ANALYSIS EXPORT COMPLETE
Files created: 9
STEP 7A: PASSED


In [12]:
# STEP 7B: Generate scenario analysis executive summary

summary_lines = [
    "## Final Summary — Disruption Scenario Analysis",
    "",
    "### Objective",
    "Evaluate supply network resilience under four",
    "hypothetical operational disruptions.",
    "",
    "### Reference Performance",
    f"- Optimized cost: **${reference_cost:,.2f}**",
    "- Original demand: **6,180 units**",
    "",
    "### Scenario Results"
]

for row in scenario_comparison.itertuples(index=False):

    if row.Status == "Optimal":
        summary_lines.append(
            f"- **{row.Scenario}**: "
            f"${row.Optimized_Cost_USD:,.2f}; "
            f"incremental cost "
            f"${row.Incremental_Cost_USD:,.2f}; "
            f"demand {row.Total_Demand_Units:,.0f} units"
        )
    else:
        summary_lines.append(
            f"- **{row.Scenario}**: {row.Status}; "
            "full-service feasibility not established"
        )

summary_lines.extend([
    "",
    "### Methodology",
    "- Rebuilt the 12-week Pyomo model for every scenario.",
    "- Applied scenario-specific changes to constraints",
    "  or cost parameters.",
    "- Reoptimized purchasing, production, shipments,",
    "  and inventory decisions.",
    "- Verified optimal solutions and independently",
    "  replayed inventory flows.",
    "",
    "### Interpretation",
    "Incremental costs are measured against the original",
    "optimized network. Demand-surge costs include the",
    "cost of additional customer units, so they must",
    "not be interpreted solely as disruption penalties.",
    "",
    "### Limitations",
    "- Hypothetical supplier and manufacturing capacities",
    "- Deterministic demand and lead times",
    "- No probabilistic disruption estimates",
    "- Full service and fixed terminal inventory required",
    "- Supplier outage represents an order-acceptance",
    "  interruption, not cancellation of prior deliveries",
    "",
    "### Next Steps",
    "1. Perform capacity and cost sensitivity analysis.",
    "2. Evaluate mitigation strategies and marginal value.",
    "3. Develop Tableau executive dashboards."
])

summary = "\n".join(summary_lines)

display(Markdown(summary))

docs_dir = repo_root / "docs"
docs_dir.mkdir(parents=True, exist_ok=True)

summary_file = docs_dir / "notebook07_findings.md"
summary_file.write_text(summary, encoding="utf-8")

assert summary_file.is_file()

print("\nExecutive summary saved:", summary_file)
print("NOTEBOOK 07: COMPLETE")


## Final Summary — Disruption Scenario Analysis

### Objective
Evaluate supply network resilience under four
hypothetical operational disruptions.

### Reference Performance
- Optimized cost: **$404,131.20**
- Original demand: **6,180 units**

### Scenario Results
- **Reference**: $404,131.20; incremental cost $0.00; demand 6,180 units
- **Supplier_Disruption**: $417,631.20; incremental cost $13,500.00; demand 6,180 units
- **Demand_Surge**: $418,967.70; incremental cost $14,836.50; demand 6,336 units
- **Manufacturing_Disruption**: $405,741.20; incremental cost $1,610.00; demand 6,180 units
- **Freight_Inflation**: $413,920.45; incremental cost $9,789.25; demand 6,180 units

### Methodology
- Rebuilt the 12-week Pyomo model for every scenario.
- Applied scenario-specific changes to constraints
  or cost parameters.
- Reoptimized purchasing, production, shipments,
  and inventory decisions.
- Verified optimal solutions and independently
  replayed inventory flows.

### Interpretation
Incremental costs are measured against the original
optimized network. Demand-surge costs include the
cost of additional customer units, so they must
not be interpreted solely as disruption penalties.

### Limitations
- Hypothetical supplier and manufacturing capacities
- Deterministic demand and lead times
- No probabilistic disruption estimates
- Full service and fixed terminal inventory required
- Supplier outage represents an order-acceptance
  interruption, not cancellation of prior deliveries

### Next Steps
1. Perform capacity and cost sensitivity analysis.
2. Evaluate mitigation strategies and marginal value.
3. Develop Tableau executive dashboards.


Executive summary saved: /Users/sidkulkarni/Desktop/supply-chain-portfolio/hardware-supply-network-optimizer/docs/notebook07_findings.md
NOTEBOOK 07: COMPLETE
